In [0]:
%sql
CREATE VOLUME fraud_detection.silver.chk_volume;

In [0]:
spark.sql("USE CATALOG fraud_detection")
spark.sql("USE SCHEMA silver")

In [0]:
from pyspark.sql.functions import *

silver_df = spark.readStream.table("fraud_detection.bronze.transactions_bronze") \
    .withColumn("txn_hour", hour(col("timestamp"))) \
    .withColumn("is_high_amount", col("amount") > 50000) \
    .withColumn("is_night_txn", col("txn_hour").between(0,4))

silver_df.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", "/Volumes/fraud_detection/silver/chk_volume/transactions_chk_fin") \
    .trigger(availableNow=True) \
    .toTable("Transactions_silver")

In [0]:
%sql
select count(*) from Transactions_silver